## Generate data

In [ ]:
import numpy as np
import rioxarray  # noqa: F401
import xarray as xra

NPIXELS_Y = 100
NPIXELS_X = 100
MIN_X = -180
MAX_X = 180
MIN_Y = -90
MAX_Y = 90
RES_X = (MAX_X - MIN_X) / NPIXELS_X
RES_Y = (MAX_Y - MIN_Y) / NPIXELS_Y

# Coordinates at pixel centers
y_coords = np.linspace(MAX_Y - RES_Y / 2, MIN_Y + RES_Y / 2, NPIXELS_Y)
x_coords = np.linspace(MIN_X + RES_X / 2, MAX_X - RES_X / 2, NPIXELS_X)
x_grid, y_grid = np.meshgrid(x_coords, y_coords)

data = ((x_grid - x_grid.min()) + (y_grid - y_grid.min())) / 2  # Diagonal gradient
data = data / data.max()  # Normalize to 0-1

da = xra.DataArray(
    data,
    dims=["latitude", "longitude"],
    coords={
        "latitude": y_coords,
        "longitude": x_coords,
    },
)
da.rio.write_crs("EPSG:4326", inplace=True)
da

In [ ]:
import cartopy.crs as ccrs
import matplotlib.pyplot as plt

fig = plt.figure(figsize=(10, 6))
ax = fig.add_subplot(1, 1, 1, projection=ccrs.Orthographic(central_longitude=160))

da.plot.pcolormesh(ax=ax, transform=ccrs.PlateCarree())

ax.coastlines()
ax.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False)
ax.set_global()

plt.show()

## Xpublish vs TiTiler

TiTiler is OK with coordinates named `y`/`x`, but Xpublish is not. Conversely, Xpublish is OK with coordinates named `lat`/`lon`, but TiTiler is not.

Both are OK with `latitude`/`longitude`. 🤷

See: 

* <https://github.com/earth-mover/xpublish-tiles/issues/206>
* <https://github.com/developmentseed/titiler/issues/1339>

### Reverse y-coordinates to conventional order

If the y-coordinates are ascending, the data is upside downsies. You'd need to:

```python
da = da.reindex(Y=da.Y[::-1])
```

### Add a geotransform?

Depending on your coordinate names, Xpublish may need more information to display your data.

For example, it was unhappy with `y`/`x` as coordinate names, which is why they are `lat`/`lon` in this example. To make other coordinate names work, you may need to set a GeoTransform on the `spatial_ref` attribute as below:

```python
from affine import Affine

transform = Affine.translation(MIN_X, MAX_Y) * Affine.scale(RES_X, -RES_Y)
da.rio.write_transform(transform, inplace=True)
```


## Render

This still renders all weird with xpublish. Some tiles don't display depending on zoom level. Wat.

<https://github.com/earth-mover/xpublish-tiles/issues/206>

In [ ]:
from jupyter_xarray_tiler.xpublish import add_data_array

url = await add_data_array(da, rescale=(0, 1))

In [ ]:
url

In [ ]:
import leafmap

m = leafmap.Map(zoom=4)
m

In [ ]:
m.add_tile_layer(
    url=url,
    name="Mock data",
    attribution="¯\\_(ツ)_/¯",
)

In [ ]:
da.spatial_ref